# HydraNet Quick Start

This notebook demonstrates:
1. Loading pre-trained model weights from HuggingFace
2. Splitting the model into ENCODER, BOTTLENECK, and DECODER components
3. Exporting ENCODER, BOTTLENECK, and DECODER to ONNX

**Repository:** `sirbastiano/hydranet-phisat2`  
**Available:** 139 pre-trained model configurations

## 1. Load Model with Pre-trained Weights

Load the student model with automatic weight download from HuggingFace.

In [1]:
import hydranet

# Configuration
task = 'burned_area'
n_shots = 5000
training = 'linear_probing'

# Load model with automatic weight download and loading
model = hydranet.load_student(
    preset='checkpoint',      # Matches HF checkpoint architecture
    task=task,
    n_shots=n_shots,
    training=training,
    auto_load_weights=True,
    weights_dir='../weights',
    strict=False             # Allows task-specific head mismatches
)

print(f"✓ Model loaded: {sum(p.numel() for p in model.parameters()):,} parameters")

Found latest checkpoint from 20251216
  Downloading: UNet_Myriad2_Downstream_frozen_best.pt
  Saved to: ../weights/linear_probing/hydranet/burned_area_nshot5000_frozen/burned_area/20251216_UNet_Myriad2_Downstream_frozen_5000/UNet_Myriad2_Downstream_frozen_best.pt
Loading weights from: ../weights/linear_probing/hydranet/burned_area_nshot5000_frozen/burned_area/20251216_UNet_Myriad2_Downstream_frozen_5000/UNet_Myriad2_Downstream_frozen_best.pt
  Adjusted n_classes to checkpoint head: 4
✓ Model loaded: 351,172 parameters


### Checkpoint Compatibility Notes

When loading weights from Hugging Face checkpoints (`auto_load_weights=True`), HydraNet applies compatibility fixes:

- Legacy checkpoint head keys `classifier.weight` / `classifier.bias` are mapped to `final_conv.weight` / `final_conv.bias`.
- If `n_classes` is not explicitly set and checkpoint head classes differ from the local model head, `final_conv` is automatically resized to match the checkpoint output classes.
- Missing `*.convnext_block.gamma` tensors in older checkpoints are filled from model defaults.

This ensures component exports (encoder, bottleneck, decoder) use the same loaded weights as the checkpoint-compatible model.


## 2. Split Model into Components

Decompose the model into ENCODER, BOTTLENECK, and DECODER components.

In [2]:
# Split the model into components
components = hydranet.split_model(model)

# Display summary
print(components)

ModelComponents(
  ENCODER: 2 layer groups, 52,304 parameters (14.9%)
  BOTTLENECK: 1 layer groups, 146,816 parameters (41.8%)
  DECODER: 3 layer groups, 152,052 parameters (43.3%)
  TOTAL: 351,172 parameters
)


In [3]:
# Access individual components
encoder = components.encoder
bottleneck = components.bottleneck
decoder = components.decoder

print("ENCODER:", list(encoder.keys()))
print("BOTTLENECK:", list(bottleneck.keys()))
print("DECODER:", list(decoder.keys()))

ENCODER: ['encoders', 'pools']
BOTTLENECK: ['bottleneck']
DECODER: ['upsamplers', 'decoders', 'final_conv']


In [4]:
encoder

ModuleDict(
  (encoders): ModuleList(
    (0): ConvBlock(
      (channel_proj): Conv2d(8, 16, kernel_size=(1, 1), stride=(1, 1))
      (convnext_block): ConvNeXtBlock(
        (dwconv): Conv2d(16, 16, kernel_size=(7, 7), stride=(1, 1), padding=(3, 3), groups=16)
        (norm): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (pwconv1): Conv2d(16, 64, kernel_size=(1, 1), stride=(1, 1))
        (act): GELU(approximate='none')
        (pwconv2): Conv2d(64, 16, kernel_size=(1, 1), stride=(1, 1))
        (drop_path): Identity()
      )
    )
    (1): ConvBlock(
      (channel_proj): Conv2d(16, 32, kernel_size=(1, 1), stride=(1, 1))
      (convnext_block): ConvNeXtBlock(
        (dwconv): Conv2d(32, 32, kernel_size=(7, 7), stride=(1, 1), padding=(3, 3), groups=32)
        (norm): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (pwconv1): Conv2d(32, 128, kernel_size=(1, 1), stride=(1, 1))
        (act): GELU(appro

## 3. Export Components to ONNX

Export each component (`encoder`, `bottleneck`, `decoder`) as a standalone ONNX model.

In [5]:
from pathlib import Path
import torch
import torch.nn as nn
import torch.nn.functional as F

output_dir = Path('../onnx/components')
output_dir.mkdir(parents=True, exist_ok=True)

model.eval()

# components.* are ModuleDict objects, so we expose explicit forward() wrappers
class EncoderExportWrapper(nn.Module):
    def __init__(self, encoders, pools):
        super().__init__()
        self.encoders = encoders
        self.pools = pools

    def forward(self, x):
        skips = []
        current = x
        for i, enc in enumerate(self.encoders):
            current = enc(current)
            skips.append(current)
            if i < len(self.encoders) - 1:
                current = self.pools[i](current)
        bottleneck_in = self.pools[-1](current)
        return (bottleneck_in, *skips)


class DecoderExportWrapper(nn.Module):
    def __init__(self, upsamplers, decoders, final_conv):
        super().__init__()
        self.upsamplers = upsamplers
        self.decoders = decoders
        self.final_conv = final_conv

    def forward(self, bottleneck_out, *skips):
        current = bottleneck_out
        depth = len(self.decoders)
        for i in range(depth):
            current = self.upsamplers[i](current)
            skip = skips[depth - 1 - i]
            if current.shape[2:] != skip.shape[2:]:
                current = F.interpolate(current, size=skip.shape[2:], mode='bilinear', align_corners=False)
            current = torch.cat([current, skip], dim=1)
            current = self.decoders[i](current)
        return self.final_conv(current)


encoder_wrapper = EncoderExportWrapper(encoder['encoders'], encoder['pools']).eval()
bottleneck_wrapper = bottleneck['bottleneck'].eval()
decoder_wrapper = DecoderExportWrapper(
    decoder['upsamplers'],
    decoder['decoders'],
    decoder['final_conv'],
).eval()

dummy_input = torch.randn(1, 8, 224, 224)

with torch.no_grad():
    encoder_outputs = encoder_wrapper(dummy_input)
    bottleneck_input = encoder_outputs[0]
    skips = encoder_outputs[1:]
    bottleneck_output = bottleneck_wrapper(bottleneck_input)

component_paths = {
    'encoder': output_dir / 'encoder.onnx',
    'bottleneck': output_dir / 'bottleneck.onnx',
    'decoder': output_dir / 'decoder.onnx',
}

# 1) Encoder export: output = bottleneck_input + all skip tensors
torch.onnx.export(
    encoder_wrapper,
    dummy_input,
    str(component_paths['encoder']),
    export_params=True,
    opset_version=17,
    input_names=['input'],
    output_names=['bottleneck_input'] + [f'skip_{i}' for i in range(len(skips))],
    dynamic_axes={
        'input': {0: 'batch_size'},
        'bottleneck_input': {0: 'batch_size'},
        **{f'skip_{i}': {0: 'batch_size'} for i in range(len(skips))},
    },
)

# 2) Bottleneck export: input = bottleneck_input, output = bottleneck_output
torch.onnx.export(
    bottleneck_wrapper,
    bottleneck_input,
    str(component_paths['bottleneck']),
    export_params=True,
    opset_version=17,
    input_names=['input'],
    output_names=['output'],
    dynamic_axes={'input': {0: 'batch_size'}, 'output': {0: 'batch_size'}},
)

# 3) Decoder export: inputs = bottleneck_output + skip tensors
decoder_input_names = ['bottleneck_output'] + [f'skip_{i}' for i in range(len(skips))]
decoder_dynamic_axes = {name: {0: 'batch_size'} for name in decoder_input_names}
decoder_dynamic_axes['output'] = {0: 'batch_size'}

torch.onnx.export(
    decoder_wrapper,
    (bottleneck_output, *skips),
    str(component_paths['decoder']),
    export_params=True,
    opset_version=10,
    input_names=decoder_input_names,
    output_names=['output'],
    dynamic_axes=decoder_dynamic_axes,
)

for name, path in component_paths.items():
    print(f'{name}: {path.resolve()}')


encoder: /Users/roberto.delprete/Library/CloudStorage/OneDrive-ESA/Desktop/Repos/phisat2/Paper/onnx/components/encoder.onnx
bottleneck: /Users/roberto.delprete/Library/CloudStorage/OneDrive-ESA/Desktop/Repos/phisat2/Paper/onnx/components/bottleneck.onnx
decoder: /Users/roberto.delprete/Library/CloudStorage/OneDrive-ESA/Desktop/Repos/phisat2/Paper/onnx/components/decoder.onnx


/var/folders/21/4frhlz5x0tb46dk9rnx9jw4884jv22/T/ipykernel_87930/1378663405.py:43: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if current.shape[2:] != skip.shape[2:]:
